# 01 Создание датасета 


In [69]:
import torch
import torch.nn as nn
import yfinance as yf
import pandas as pd
import torch.optim as optim
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import random
import warnings
import os
from numpy import random as rnp

from plotly.io import templates
templates.default = "plotly_dark"

torch.manual_seed(424242)
torch.cuda.manual_seed(424242)
rnp.seed(424242)
random.seed(424242)

In [70]:
from datetime import date
from torch.utils.data import Dataset, DataLoader

In [71]:
from uberdataset import UberDatasetFuhrer

## Параметры 

In [ ]:
TICKERS = ['BTC-USD', 'ETH-USD', 'SOL-USD', 'BNB-USD', 'XRP-USD', 'AVAX-USD']
# TICKERS = ['BTC-USD']
# TICKERS = [
#     'BTC-USD', 'ETH-USD', 'BNB-USD', 'XRP-USD', 'SOL-USD',
#     'ADA-USD', 'DOGE-USD', 'MATIC-USD', 'AVAX-USD', 'SHIB-USD',
#     'LINK-USD', 'DOT-USD', 'LTC-USD', 'BCH-USD', 'UNI-USD', 'ICP-USD',
#     'TON-USD', 'FTM-USD', 'NEAR-USD', 'QNT-USD', 'ALGO-USD', 'SUI-USD',
#     'ATOM-USD', 'XLM-USD', 'FIL-USD', 'EGLD-USD', 'EOS-USD', 'AAVE-USD',
#     'GRT-USD', 'MANA-USD', 'AXS-USD', 'HBAR-USD', 'THETA-USD', 'CHZ-USD',
#     'CAKE-USD', 'ENJ-USD', 'SNX-USD', 'YFI-USD', 'ZEC-USD', 'XMR-USD',
#     'COMP-USD', 'BAT-USD', 'ZRX-USD', 'SAND-USD', 'LDO-USD',
#     'FET-USD', 'LRC-USD', 'AR-USD', 'APT-USD', 'STX-USD',
# ]
START_DATE = "2019-01-01"
END_DATE = "2026-03-11"

## 1. Загрузка данных

In [73]:
raw = yf.download(
    TICKERS,
    start=START_DATE,
    end=END_DATE,
    group_by="ticker",
    interval="1d",
    threads=True,
)

REQUIRED_COLS = ['Close', 'High', 'Low', 'Open', 'Volume']


def _normalize_multiindex(raw_df: pd.DataFrame, tickers: list[str]) -> pd.DataFrame:
    """
    Ensure columns are (ticker, field) regardless of yfinance version.

    Older yfinance with group_by='ticker' returns (ticker, field).
    Newer yfinance may return (field, ticker) — detect and swap if needed.
    """
    if not isinstance(raw_df.columns, pd.MultiIndex):
        return raw_df
    level0_samples = set(raw_df.columns.get_level_values(0).unique())
    price_fields = {'Close', 'High', 'Low', 'Open', 'Volume', 'Adj Close'}
    if level0_samples & price_fields:
        raw_df = raw_df.swaplevel(axis=1).sort_index(axis=1)
    return raw_df


def extract_ticker_frames(raw_df: pd.DataFrame, tickers: list[str]) -> dict[str, pd.DataFrame]:
    """Split the multi-ticker download into clean per-ticker DataFrames."""
    raw_df = _normalize_multiindex(raw_df, tickers)
    frames: dict[str, pd.DataFrame] = {}

    for ticker in tickers:
        try:
            df = pd.DataFrame(raw_df[ticker])[REQUIRED_COLS].copy()
        except KeyError:
            warnings.warn(f'{ticker}: missing columns, skipping')
            continue

        # Drop rows where ALL OHLCV are NaN (weekends / non-trading days)
        df.dropna(how='all', inplace=True)

        # Forward-fill isolated gaps, then drop any remaining NaNs at the start
        df.ffill(inplace=True)
        df.dropna(inplace=True)

        if len(df) < 300:
            warnings.warn(f'{ticker}: only {len(df)} rows after cleaning, skipping')
            continue

        df.sort_index(ascending=True, inplace=True)
        frames[ticker] = df

    return frames

ticker_frames = extract_ticker_frames(raw, TICKERS)
print(f'Successfully loaded {len(ticker_frames)} / {len(TICKERS)} tickers')
print(f'Tickers: {list(ticker_frames.keys())}')

[*********************100%***********************]  6 of 6 completed

Successfully loaded 6 / 6 tickers
Tickers: ['BTC-USD', 'ETH-USD', 'SOL-USD', 'BNB-USD', 'XRP-USD', 'AVAX-USD']


## 2. Определения признаков

In [74]:
EPS = 1e-8

# ── Shared constants ──────────────────────────────────────────────────
return_horizons = (1, 3, 5, 10, 20)
log_close_return_columns = [f'log_close_return_{h}' for h in return_horizons]
log_low_return_columns   = [f'log_low_return_{h}' for h in return_horizons]
log_high_return_columns  = [f'log_high_return_{h}' for h in return_horizons]
log_open_return_columns  = [f'log_open_return_{h}' for h in return_horizons]

log_return_columns = (
    log_close_return_columns
    + log_low_return_columns
    + log_high_return_columns
    + log_open_return_columns
)

vol_windows = (5, 10, 20, 60)
vol_columns  = [f'vol_{w}' for w in vol_windows]

dev_windows  = (10, 20, 50, 100)
dev_columns  = [f'dev_{w}' for w in dev_windows]

range_columns          = ['range']
close_position_columns = ['close_position']
log_volume_columns     = ['log_volume']
volume_return_columns  = ['volume_return']
volume_z20_columns     = ['volume_z20']
rsi_columns            = ['rsi_14']
macd_columns           = ['macd_hist']
roc10_columns          = ['roc_10']
entropy_columns        = ['entropy']
skew_columns           = ['skew']
kurt_columns           = ['kurt']
bull_regime_columns    = ['bull_regime']
high_vol_regime_columns= ['high_vol_regime']

# ── NEW: Tier 2 column constants ─────────────────────────────────────
bb_pct_b_columns       = ['bb_pct_b']
ret_autocorr_columns   = ['ret_autocorr']
vol_ratio_columns      = ['vol_ratio']
atr_norm_columns       = ['atr_norm']
dow_columns            = ['dow_sin', 'dow_cos']  # cyclical, NOT z-scored

macd_fast, macd_slow, macd_signal_w = 12, 26, 9
entropy_window = 20
standardization_window = 100

standardization_deps = [
    'log_returns', 'volatility', 'trend_deviation', 'range',
    'close_position', 'log_volume', 'volume_return', 'rsi_14',
    'macd_hist', 'roc_10', 'entropy', 'skew', 'kurt', 'volume_z20',
    # NEW deps for tier 2
    'bb_pct_b', 'ret_autocorr', 'vol_ratio', 'atr_norm',
]

standardization_in_columns = (
    log_return_columns + vol_columns + dev_columns + range_columns
    + close_position_columns + log_volume_columns + volume_return_columns
    + rsi_columns + macd_columns + roc10_columns
    + entropy_columns + skew_columns + kurt_columns + volume_z20_columns
    # NEW: tier 2 raw columns to z-score
    + bb_pct_b_columns + ret_autocorr_columns + vol_ratio_columns
    + atr_norm_columns
)
standardization_out_columns = [f'z_{c}' for c in standardization_in_columns]


def entropy_of_window(x: np.ndarray) -> float:
    x = x[np.isfinite(x)]
    if x.size < 5:
        return np.nan
    hist, _ = np.histogram(x, bins=10, density=True)
    p = hist.astype(np.float64)
    p = p[p > 0]
    if p.size == 0:
        return np.nan
    p = p / p.sum()
    return float(-(p * np.log(p + EPS)).sum())


def register_features(ds: UberDatasetFuhrer) -> None:
    """Register all feature functions on *ds* (uses programmatic API)."""

    # 1. Log returns
    def ft_log_returns(df):
        for h, name in zip(return_horizons, log_close_return_columns):
            df[name] = np.log(df['Close'] / df['Close'].shift(h) + EPS)

        for h, name in zip(return_horizons, log_low_return_columns):
            df[name] = np.log(df['Low'] / df['Low'].shift(h) + EPS)           
            
        for h, name in zip(return_horizons, log_high_return_columns):
            df[name] = np.log(df['High'] / df['High'].shift(h) + EPS)

        for h, name in zip(return_horizons, log_open_return_columns):
            df[name] = np.log(df['Open'] / df['Open'].shift(h) + EPS)
            
    ds.register(ft_log_returns, name='log_returns', produces=log_return_columns)

    # 2. Volatility (depends on log_returns for log_close_return_1)
    def ft_volatility(df):
        for w, name in zip(vol_windows, vol_columns):
            df[name] = df['log_close_return_1'].rolling(w).std()
    ds.register(ft_volatility, name='volatility', deps=['log_returns'], produces=vol_columns)

    # 3. Trend deviation
    def ft_trend_deviation(df):
        for w, name in zip(dev_windows, dev_columns):
            ma = df['Close'].rolling(w).mean()
            df[name] = (df['Close'] - ma) / (ma + EPS)
    ds.register(ft_trend_deviation, name='trend_deviation', produces=dev_columns)

    # 4. Candle structure
    def ft_range(df):
        df['range'] = (df['High'] - df['Low']) / (df['Close'] + EPS)
    ds.register(ft_range, name='range', produces=range_columns)

    def ft_close_position(df):
        denom = (df['High'] - df['Low']).replace(0, np.nan)
        df['close_position'] = ((df['Close'] - df['Low']) / denom).clip(0.0, 1.0)
    ds.register(ft_close_position, name='close_position', produces=close_position_columns)

    # 5. Volume features
    def ft_log_volume(df):
        df['log_volume'] = np.log(df['Volume'] + 1.0)
    ds.register(ft_log_volume, name='log_volume', produces=log_volume_columns)

    def ft_volume_return(df):
        df['volume_return'] = np.log((df['Volume'] + 1.0) / (df['Volume'].shift(1) + 1.0))
    ds.register(ft_volume_return, name='volume_return', produces=volume_return_columns)

    def ft_volume_z20(df):
        rm = df['Volume'].rolling(20).mean()
        rs = df['Volume'].rolling(20).std().replace(0, np.nan) + EPS
        df['volume_z20'] = (df['Volume'] - rm) / rs
    ds.register(ft_volume_z20, name='volume_z20', produces=volume_z20_columns)

    # 6. Momentum
    def ft_rsi_14(df):
        delta = df['Close'].diff()
        gain  = delta.clip(lower=0.0)
        loss  = (-delta).clip(lower=0.0)
        avg_gain = gain.rolling(14).mean()
        avg_loss = loss.rolling(14).mean()
        rs = avg_gain / (avg_loss + EPS)
        df['rsi_14'] = 100.0 - (100.0 / (1.0 + rs))
    ds.register(ft_rsi_14, name='rsi_14', produces=rsi_columns)

    def ft_macd_hist(df):
        ema_f = df['Close'].ewm(span=macd_fast, adjust=False).mean()
        ema_s = df['Close'].ewm(span=macd_slow, adjust=False).mean()
        macd  = ema_f - ema_s
        sig   = macd.ewm(span=macd_signal_w, adjust=False).mean()
        df['macd_hist'] = macd - sig
    ds.register(ft_macd_hist, name='macd_hist', produces=macd_columns)

    def ft_roc_10(df):
        df['roc_10'] = df['Close'].pct_change(10)
    ds.register(ft_roc_10, name='roc_10', produces=roc10_columns)

    # 7. Distribution-based
    def ft_entropy(df):
        df['entropy'] = (
            df['log_close_return_1']
            .rolling(entropy_window)
            .apply(lambda s: entropy_of_window(s.to_numpy()), raw=False)
        )
    ds.register(ft_entropy, name='entropy', deps=['log_returns'], produces=entropy_columns)

    def ft_skew(df):
        df['skew'] = df['entropy'].rolling(entropy_window).skew()
    ds.register(ft_skew, name='skew', deps=['entropy'], produces=skew_columns)

    def ft_kurt(df):
        df['kurt'] = df['entropy'].rolling(entropy_window).kurt()
    ds.register(ft_kurt, name='kurt', deps=['entropy'], produces=kurt_columns)

    # 8. Regime flags
    def ft_bull_regime(df):
        ma50  = df['Close'].rolling(50).mean()
        ma200 = df['Close'].rolling(200).mean()
        df['bull_regime'] = (ma50 > ma200).astype(int)
    ds.register(ft_bull_regime, name='bull_regime', produces=bull_regime_columns)

    def ft_high_vol_regime(df):
        sigma = df['vol_20']
        df['high_vol_regime'] = (sigma > sigma.rolling(200).mean()).astype(int)
    ds.register(ft_high_vol_regime, name='high_vol_regime', deps=['volatility'], produces=high_vol_regime_columns)

    # ── 8b. NEW: Tier 2 features ─────────────────────────────────────

    # Bollinger Band %B: where price sits within the bands (mean-reversion signal)
    # 0 = lower band, 0.5 = middle, 1 = upper band
    # Corr 0.85 with RSI — partially redundant but captures price-vs-distribution
    def ft_bb_pct_b(df):
        sma = df['Close'].rolling(20).mean()
        std = df['Close'].rolling(20).std()
        upper = sma + 2.0 * std
        lower = sma - 2.0 * std
        df['bb_pct_b'] = (df['Close'] - lower) / (upper - lower + EPS)
    ds.register(ft_bb_pct_b, name='bb_pct_b', produces=bb_pct_b_columns)

    # Rolling lag-1 autocorrelation of returns (corr <0.16 with everything — pure new info)
    # Positive = trending, negative = mean-reverting, ~0 = random walk
    def ft_ret_autocorr(df):
        df['ret_autocorr'] = (
            df['log_close_return_1']
            .rolling(20)
            .apply(lambda x: pd.Series(x).autocorr(lag=1), raw=False)
        )
    ds.register(ft_ret_autocorr, name='ret_autocorr', deps=['log_returns'], produces=ret_autocorr_columns)

    # Volatility ratio: short-term vol / long-term vol (corr 0.44 — good)
    # >1 = vol expanding (breakout), <1 = vol compressing (calm before storm)
    def ft_vol_ratio(df):
        vol_fast = df['log_close_return_1'].rolling(5).std()
        vol_slow = df['log_close_return_1'].rolling(60).std()
        df['vol_ratio'] = vol_fast / (vol_slow + EPS)
    ds.register(ft_vol_ratio, name='vol_ratio', deps=['log_returns'], produces=vol_ratio_columns)

    # Normalized ATR: intraday range expansion the close-to-close returns miss
    def ft_atr_norm(df):
        prev_close = df['Close'].shift(1)
        tr = pd.concat([
            df['High'] - df['Low'],
            (df['High'] - prev_close).abs(),
            (df['Low']  - prev_close).abs(),
        ], axis=1).max(axis=1)
        df['atr_norm'] = tr.rolling(14).mean() / (df['Close'] + EPS)
    ds.register(ft_atr_norm, name='atr_norm', produces=atr_norm_columns)

    # Day-of-week cyclical encoding (corr ~0.02 — pure new info)
    # NOT z-scored: already bounded [-1, 1]
    def ft_day_of_week(df):
        day = pd.DatetimeIndex(df.index).dayofweek.astype(float)
        df['dow_sin'] = np.sin(2 * np.pi * day / 7)
        df['dow_cos'] = np.cos(2 * np.pi * day / 7)
    ds.register(ft_day_of_week, name='day_of_week', produces=dow_columns)

    # ── END Tier 2 ───────────────────────────────────────────────────

    # 9. Target
    def ft_target(df):
        df['target'] = np.log(df['Close'].shift(-1) / df['Close'] + EPS)
    ds.register(ft_target, name='target', produces=['target'])

    # 10. Rolling standardization (now includes tier 2 raw columns)
    def ft_standardization(df):
        for in_col, out_col in zip(standardization_in_columns, standardization_out_columns):
            rm = df[in_col].rolling(standardization_window).mean()
            rs = df[in_col].rolling(standardization_window).std().replace(0, np.nan)
            df[out_col] = (df[in_col] - rm) / (rs + EPS)

    ds.register(
        ft_standardization,
        name='standardization',
        deps=standardization_deps,
        produces=standardization_out_columns,
    )

     # 11. Scale target by target_std
    def ft_target_scaling(df):
        target_std = df['target'].rolling(standardization_window).std().replace(0, np.nan)
        target_mu = df['target'].rolling(standardization_window).mean()
        df['target'] = (df['target'] - target_mu) / (target_std + EPS)
        df['target_scale'] = target_std
        df['target_mu'] = target_mu
    
    ds.register(
        ft_target_scaling,
        name='target_scaling',
        deps=['target', 'standardization'],
        produces=['target', 'target_scale', 'target_mu'],
    )

## 3. Построение признаков для каждого тикера

In [75]:
built_frames: list[pd.DataFrame] = []
failed_tickers: list[tuple[str, str]] = []

for ticker, frame in ticker_frames.items():
    try:
        ds = UberDatasetFuhrer(frame, feature_fn_prefix='ft_')
        register_features(ds)
        ds.build_all()

        if ds.df.empty:
            failed_tickers.append((ticker, 'empty after build (too few rows for rolling windows)'))
            print(f'  ✗ {ticker:12s}  empty after dropna')
            continue

        ds.df['ticker'] = ticker
        built_frames.append(ds.df)
        print(f'  ✓ {ticker:12s}  rows={len(ds.df):>6,}')
    except Exception as e:
        failed_tickers.append((ticker, str(e)))
        print(f'  ✗ {ticker:12s}  {e}')

print(f'\nBuilt {len(built_frames)} tickers, {len(failed_tickers)} failed')
if failed_tickers:
    print('\nFailed tickers:')
    for t, reason in failed_tickers:
        print(f'  {t}: {reason}')

  ✓ BTC-USD       rows= 2,428
  ✓ ETH-USD       rows= 2,428
  ✓ SOL-USD       rows= 1,963
  ✓ BNB-USD       rows= 2,428
  ✓ XRP-USD       rows= 2,428
  ✓ AVAX-USD      rows= 1,800

Built 6 tickers, 0 failed


## 4. Объединение в единый датасет

In [76]:
assert len(built_frames) > 0, (
    f'No tickers built successfully! All {len(failed_tickers)} failed. '
    f'Check the error messages above.'
)

combined_df = pd.concat(built_frames, axis=0)
combined_df.sort_index(ascending=True, inplace=True)
print(f'Combined shape: {combined_df.shape}')
print(f'Date range: {combined_df.index.min()} → {combined_df.index.max()}')
print(f'Tickers: {combined_df["ticker"].nunique()}')
combined_df.head()

Combined shape: (13475, 99)
Date range: 2019-07-18 00:00:00 → 2026-03-10 00:00:00
Tickers: 6


Price,Close,High,Low,Open,Volume,atr_norm,bb_pct_b,bull_regime,close_position,dow_sin,...,z_kurt,z_volume_z20,z_bb_pct_b,z_ret_autocorr,z_vol_ratio,z_atr_norm,target,target_scale,target_mu,ticker
Date,,,,,,,,,,,,,,,,,,,,,
2019-07-18,10666.482422,10736.842773,9376.798828,9698.502930,2.518702e+10,0.087834,0.344807,0,0.948266,0.433884,...,-0.331113,-0.292175,-1.412577,-0.239888,1.743374,1.490398,-0.403543,0.048640,0.006820,BTC-USD
2019-07-18,226.566162,229.239395,208.037735,211.444611,9.327816e+09,0.092536,0.166772,0,0.873914,0.433884,...,1.347703,-0.062971,-1.297649,-0.279497,2.761851,1.960306,-0.509939,0.050169,0.002216,ETH-USD
2019-07-18,28.922812,29.159201,26.937096,27.257734,7.118958e+08,0.071452,0.253890,0,0.893619,0.433884,...,0.127731,2.181219,-1.001943,-0.545934,2.748665,0.935742,0.015087,0.046778,0.004659,BNB-USD
2019-07-18,0.323176,0.326506,0.306624,0.310748,1.460358e+09,0.075590,0.243264,0,0.832512,0.433884,...,-0.487758,-0.317649,-0.745909,-0.611389,0.879964,0.934385,-0.144627,0.048220,-0.001013,XRP-USD
2019-07-19,10530.732422,10716.980469,10229.628906,10653.956055,2.072743e+10,0.088740,0.323231,1,0.617836,-0.433884,...,-0.445902,-1.156708,-1.457869,-0.485842,1.241554,1.493354,0.303354,0.048322,0.007542,BTC-USD


## Количество данных для тиккеров

In [77]:
# Rows per ticker
ticker_counts = combined_df.groupby('ticker').size().sort_values(ascending=False)
print('Rows per ticker (top 10):')
print(ticker_counts.head(10))
print(f'\nTotal NaNs: {combined_df.isna().sum().sum()}')

Rows per ticker (top 10):
ticker
BNB-USD     2428
BTC-USD     2428
ETH-USD     2428
XRP-USD     2428
SOL-USD     1963
AVAX-USD    1800
dtype: int64

Total NaNs: 0


In [78]:
fig = px.bar(
    x=ticker_counts.index,
    y=ticker_counts.values,
    title='Rows per Ticker',
    labels={'x': 'Ticker', 'y': 'Rows'},
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

In [79]:
# Feature correlation heatmap (standardised features, sampled for speed)
z_cols = [c for c in combined_df.columns if c.startswith('z_')]
sample = combined_df[z_cols].sample(min(5000, len(combined_df)), random_state=42)
fig_corr = px.imshow(
    sample.corr(),
    title='Standardised Feature Correlation',
    color_continuous_scale='RdBu_r',
    zmin=-1, zmax=1,
    width=900, height=900,
)
fig_corr.show()

## 6. Сплит Train / Validation / Test

In [80]:
def temporal_split_per_ticker(
    df: pd.DataFrame,
    train: float = 0.8,
    val: float = 0.1,
    test: float = 0.1,
    key: str = '_split',
) -> pd.DataFrame:
    """Assign train/val/test labels independently per ticker (time-ordered)."""
    assert abs(train + val + test - 1.0) < 1e-9
    labels = pd.Series(index=df.index, dtype='object')

    for ticker, grp in df.groupby('ticker'):
        n = len(grp)
        n_train = int(n * train)
        n_val   = int(n * val)
        idx = grp.index  # already time-sorted

        labels.loc[idx[:n_train]]              = 'train'
        labels.loc[idx[n_train:n_train+n_val]] = 'val'
        labels.loc[idx[n_train+n_val:]]        = 'test'

    df[key] = labels
    return df

combined_df = temporal_split_per_ticker(combined_df)
print(combined_df['_split'].value_counts())

_split
train    10559
test      1464
val       1452
Name: count, dtype: int64


## 7. Сохранение сплитов в CSV

In [81]:
os.makedirs('./data', exist_ok=True)

for split_name in ('train', 'val', 'test'):
    subset = combined_df[combined_df['_split'] == split_name]
    path = f'./data/{split_name}_features.csv'
    subset.to_csv(path, index=True)
    print(f'{split_name:>5s}: {len(subset):>8,} rows  →  {path}')

train:   10,559 rows  →  ./data/train_features.csv
  val:    1,452 rows  →  ./data/val_features.csv
 test:    1,464 rows  →  ./data/test_features.csv
